# Reproducing Su, Kosillo, Jung et al. (2026) from dandiset 001893

Paper. Su, Kosillo, Jung et al., "Topographic structure and function of locus coeruleus
noradrenaline neurons", Nature (2026), doi:10.1038/s41586-026-11026-0.

Dataset. DANDI:001893, "Patch-seq profiling of mouse locus coeruleus norepinephrine neurons",
version 0.260817.1647 (doi:10.48324/dandi.001893/0.260817.1647). It is the only published
version of the dandiset (the draft has the same 105 assets and size), and it is the version
the paper's code (LCNE-patchseq-figures at 79fa20a, `code/dandi_assets.py`) reads, so it is
the version the paper cites through its code. Every file is streamed from the archive.

Text the claims were written from. The version of record (main text, Methods, figure
legends, Extended Data legends), with the bioRxiv preprint read only to compare; see the
opening comment of `claims/001893/10.1038-s41586-026-11026-0.yaml`.

Outside data. The Patch-seq FASTQ files at the NeMO Archive (collection nemo:col-s96tk9a,
each file pinned by its NeMO identifier and MD5 sum), for `patchseq-ne-identity`; the frozen
time-constant table of LCNE-patchseq-figures at 79fa20a, for the code variant of the tau
claims; the mm39 (GRCm39) sequence of eight gene bodies from the UCSC Genome Browser API, at
Ensembl release 116 coordinates, to count marker reads; and, to check that counting against an
aligner, the FASTQ files of 60 of the paper's retro-seq nuclei (nemo:col-3qj1ect, MD5 sums from
its manifest) with their STAR 2.7 CPM (nemo:col-afgz110).

| Claim | Tested here |
|---|---|
| patchseq-sample | tested |
| patchseq-transcriptome-link (link, added) | tested |
| patchseq-ne-identity | tested (data at NeMO) |
| tau-table-link (link, added) | tested |
| patchseq-ap-waveform-spinal-vs-cortex | tested |
| patchseq-ap-waveform-spinal-vs-cerebellum | tested |
| patchseq-tau-cortex-vs-spinal | tested |
| patchseq-tau-cortex-vs-cerebellum | tested |
| lc-definition, axon-density, axon-length-and-branching, single-neuron-primary-targets, markov-renewal-model, mapseq-consistent, dorsal-ventral-projection-targets, retrograde-dorsal-cortex-ventral-spinal | blocked, data not in the dandiset |
| snrnaseq-single-cluster | blocked, methods_underspecified: the criterion asks for the silhouette score at K = 1, which is undefined for a single cluster, and the paper does not say how it was computed |
| pseudospace-graded-genes, merfish-pseudospace-dorsal-ventral, retroseq-projection-gene-space | blocked, not_runnable: the counts are at NeMO, in the child collections of col-p9d5w39 (listed under "Blocked claims"), but each claim starts from the paper's selection of 4,743 LC-NE nuclei by an R pipeline (Seurat, Harmony), which the entry's Python environment cannot run |
| projection-and-expression-axes, behaviour-history-dependence, go-cue-response, ventral-respond-ignore, dorsal-switch-stay, cortex-input-switch, outcome-responses-spatial, rpe-predicts-choice-change, outcome-coefficients-cluster, frontal-axons-rpe, extracellular-waveform-spatial | blocked, data not in the dandiset |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import io, time, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import lcne, rna

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
print("dandiset", lcne.DANDISET_ID, "version", lcne.DANDISET_VERSION)
T0 = time.time()

## Loading

The asset list and the BIDS tables (`participants.tsv`, `mapping.tsv`, `datasets.tsv`) come
from the pinned version. Each NWB file is opened over HTTP with `remfile` and `h5py`, and
only the long-square current-clamp sweeps, two voltage-clamp test-pulse sweeps and the
metadata are read (`lcne.extract_cell`).

In [ ]:
assets = lcne.list_assets()
nwb = lcne.nwb_table(assets)
participants = lcne.read_tsv_asset(assets, "participants.tsv")
mapping = lcne.read_tsv_asset(assets, "mapping.tsv")
datasets_tsv = lcne.read_tsv_asset(assets, "datasets.tsv")
cells = nwb.merge(participants, on="participants_id", how="left")
cells = cells.merge(mapping.assign(ephys_roi_id=mapping.session_id.str.replace("ses-", "")),
                    on="ephys_roi_id", how="left", suffixes=("", "_mapping"))
t = time.time()
raw = lcne.extract_all(cells.asset_id)
failed = {k: v for k, v in raw.items() if isinstance(v, str)}
print(f"read {len(raw)} NWB files in {time.time() - t:.0f} s; failed to read: {len(failed)}", failed)

## Data check

In [ ]:
print("assets:", len(assets), "| NWB files:", len(nwb), "| other files:", sorted(set(assets.path) - set(nwb.path)))
print("unique ephys_roi_id:", nwb.ephys_roi_id.nunique(), "| subjects with files:", nwb.participants_id.nunique(),
      "| rows in participants.tsv:", len(participants), "| rows in mapping.tsv:", len(mapping))
print("\nCells and mice by injection target (participants.tsv injection_target_region):")
print(cells.groupby("injection_target_region").agg(cells=("ephys_roi_id", "size"), mice=("participants_id", "nunique")))
print("\nmapping.tsv session ids that match a file:", mapping.session_id.str.replace("ses-", "").isin(nwb.ephys_roi_id).sum(), "of", len(mapping))
# both tables have donor_local_id, so the merge keeps both (the participants' one unsuffixed)
assert {"donor_local_id", "donor_local_id_mapping"} <= set(cells.columns)
print("mapping donor_local_id equals participants donor_local_id:",
      int((cells.donor_local_id_mapping.str.strip() == cells.donor_local_id.str.strip()).sum()), "of", len(cells),
      "| missing on either side:", int((cells.donor_local_id_mapping.isna() | cells.donor_local_id.isna()).sum()))
print("mapping dandi_file_url names the file's own asset:",
      int(cells.apply(lambda r: isinstance(r.dandi_file_url, str) and r.asset_id in r.dandi_file_url, axis=1).sum()), "of", len(cells))
print("tissue_sample_brain_structure_name_ccf:", cells.tissue_sample_brain_structure_name_ccf.value_counts().to_dict())

# subject metadata: the NWB files against participants.tsv
meta = pd.DataFrame({aid: {"nwb_subject": c["subject_id"], "nwb_sex": c["subject_sex"], "nwb_age": c["subject_age"],
                           "nwb_genotype": c["subject_genotype"], "filtering": c["filtering"],
                           "start": c["session_start_time"], "digest": c["rheo_digest"]}
                     for aid, c in raw.items()}).T
cells = cells.join(meta, on="asset_id")
print("\nNWB subject_id equals the participants_id:", (("sub-" + cells.nwb_subject) == cells.participants_id).sum(), "of", len(cells))
sex_tsv = cells.sex_at_birth.str.upper().str[0]
print("NWB sex equals participants.tsv sex_at_birth:", (cells.nwb_sex.str.upper().str[0] == sex_tsv).sum(), "of", len(cells),
      "| participants with no sex or age in participants.tsv:", participants.loc[participants.sex_at_birth.isna() | participants.age_at_death.isna(), "participants_id"].tolist())
print("NWB sex/age of those subjects:", cells.loc[cells.sex_at_birth.isna(), ["participants_id", "nwb_sex", "nwb_age"]].drop_duplicates().values.tolist())
ages = participants.age_at_death.dropna().str.replace("P", "").astype(int)
print("ages (participants.tsv, days):", ages.min(), "to", ages.max(), "| sexes:", participants.sex_at_birth.value_counts(dropna=False).to_dict())
print("genotypes in the NWB files:", cells.nwb_genotype.value_counts().to_dict())
wt = cells[~cells.nwb_genotype.str.contains("Dbh")]
print("files whose genotype is not Dbh-Cre;Ai65:", wt.groupby(["participants_id", "injection_target_region"]).size().to_dict(),
      "| their sample names:", sorted(wt.dissociated_cell_sample_local_name))
print("intracellular_ephys filtering field:", cells.filtering.value_counts().to_dict())
print("\ndatasets.tsv:"); print(datasets_tsv.dropna(how="all").to_string())

# recordings deposited twice
print("\nduplicate session start times:", cells.start.duplicated().sum(),
      "| duplicate digests of the first spiking rheobase sweep:", cells.digest.dropna().duplicated().sum(),
      "| cells with no such sweep:", cells.digest.isna().sum())

# sweeps
sw_all = pd.concat([c["sweeps"].assign(ephys_roi_id=r) for r, c in zip(cells.ephys_roi_id, cells.asset_id.map(raw))])
ls = sw_all[sw_all.stimulus.isin(lcne.LONG_SQUARE_SETS)]
print("\ncurrent-clamp sweeps:", len(sw_all), "| by stimulus set:", sw_all.stimulus.value_counts().to_dict())
print("sampling rates (Hz):", sw_all.rate.value_counts().to_dict(), "| data unit attribute:", sw_all.unit.value_counts().to_dict())
print("stimulus sets of the sweeps not at 50 kHz:", sw_all[sw_all.rate != 50000].stimulus.value_counts().to_dict())
full = ls[(ls.pulse_dur_s >= lcne.MIN_PULSE_S)]
print("long-square sweeps with a main pulse >= 0.8 s:", len(full), "| pulse durations (s):", full.pulse_dur_s.round(3).value_counts().to_dict())
print("  aborted (NaN voltage within the pulse or the 0.5 s after it):", int(full.aborted.sum()))
print("  amplitudes (pA): %.0f to %.0f" % (full.amplitude_pa.min(), full.amplitude_pa.max()))
print("  voltage range over these sweeps (mV): %.1f to %.1f; baseline before the pulse %.1f to %.1f" %
      (full.v_min.min(), full.v_max.max(), full.v_base.min(), full.v_base.max()))
print("  sweeps with a voltage below -150 mV or above 80 mV anywhere:", int(((full.v_min < -150) | (full.v_max > 80)).sum()))
print("  bridge balance (MOhm): %.1f to %.1f | bias current, NWB bias_current (pA): %.1f to %.1f, missing in %d sweeps" %
      (full.bridge_balance_mohm.min(), full.bridge_balance_mohm.max(), full.bias_current_pa.min(), full.bias_current_pa.max(),
       full.bias_current_pa.isna().sum()))
print("  lab notebook, I-Clamp holding on / off / no entry:", int((full.holding_enable == 1).sum()), "/",
      int((full.holding_enable == 0).sum()), "/", int(full.holding_enable.isna().sum()),
      "| sweeps where holding is on and the NWB value equals the notebook's holding level (0.5 pA):",
      int(np.isclose(full.bias_current_pa, full.holding_level_pa, atol=0.5).sum()), "of", int((full.holding_enable == 1).sum()),
      "| sweeps with holding off that carry an NWB bias value:", int((full.holding_enable == 0).sum() - full.bias_current_pa[full.holding_enable == 0].isna().sum()),
      "| cells with a sweep with holding off:", full.loc[full.holding_enable == 0, "ephys_roi_id"].nunique())
print("  pre-stimulus noise RMS 1.5 ms (mV): %.3f to %.3f | 500 ms: %.3f to %.3f" %
      (full.pre_noise_mv.min(), full.pre_noise_mv.max(), full.slow_noise_mv.min(), full.slow_noise_mv.max()))

Stimulus and response share a clock. Each sweep's voltage and stimulus are stored in
paired series with one sampling rate. If they are aligned, the voltage change over the first
100 ms of the pulse follows the injected current, and the same measure 200 ms earlier,
before the pulse, does not.

In [ ]:
ok = full[~full.aborted & full.deflection_mv.notna()]
for lab, col in [("pulse window", "deflection_mv"), ("window shifted 200 ms earlier", "deflection_shifted_mv")]:
    x = ok.dropna(subset=[col])
    r, p = stats.spearmanr(x.amplitude_pa, x[col])
    print(f"{lab}: Spearman rho between amplitude and voltage change = {r:.3f} (p = {p:.2g}, {len(x)} sweeps); "
          f"median change for negative / positive steps = {x.loc[x.amplitude_pa < 0, col].median():.2f} / {x.loc[x.amplitude_pa > 0, col].median():.2f} mV")
per_cell = ok.groupby("ephys_roi_id").apply(lambda g: stats.spearmanr(g.amplitude_pa, g.deflection_mv)[0] if g.amplitude_pa.nunique() > 2 else np.nan)
print("cells with >2 amplitudes:", per_cell.notna().sum(), "| per-cell rho > 0.5 in", (per_cell > 0.5).sum(), "| lowest per-cell rho %.2f" % per_cell.min())

Checks of the metadata against the data. The data stored in volts with a conversion
factor read as millivolts (resting baselines of -81 to -15 mV, spike peaks up to 74 mV; 3
long-square sweeps reach values below -150 mV, artefacts), and
the long squares, current-clamp sweeps and pulse durations (all 1.0 s) are as the paper and
its code describe. All sweeps the analysis uses are sampled at 50 kHz, as the Methods state;
the 12.5 kHz sweeps are chirp stimuli, not used. The filter setting (Bessel, 10 kHz, in the
Methods) is not recorded: the NWB field reads "PLACEHOLDER". `participants.tsv` gives no sex
or age for sub-1435396971; its NWB files give female, P62. Eight cells of two spinal cord
donors carry the genotype wt/wt and C57BL6J sample names, where the Methods describe
Dbh-Cre;Ai65 mice for the Patch-seq experiment. `datasets.tsv` gives dandiset 001858 the URL
of dandiset 001748. The stimulus and the voltage share a clock: the voltage follows the
injected current over the pulse and not 200 ms before it. The NWB `bias_current` of a sweep is
the holding current the amplifier applied: in the MIES lab notebook each file holds, I-Clamp
holding is on for 2,433 of the 2,450 long squares of at least 0.8 s, and in all 2,433 its level
equals the NWB value; the other 17 sweeps have holding off and no NWB bias value, and are taken
as 0 pA (no current applied).

## patchseq-sample

> Whole-cell patch-clamp recordings were made from 96 LC-NE neurons in brain slices from 28
> mice, retrogradely labelled from frontal cortex (27 cells, 8 mice), cerebellar cortex (15
> cells, 7 mice) or spinal cord (54 cells, 13 mice).

A cell is one NWB file (one ephys_roi_id), a mouse one donor, the group the injection target.
The cell inclusion values of the Methods (seal > 1 GOhm, initial access resistance < 20 MOhm
and < 15% of input resistance) are measured with IPFX's functions on the last cell-attached
and last break-in test-pulse sweeps, and reported.

In [ ]:
counts = cells.groupby("injection_target_region").agg(cells=("ephys_roi_id", "nunique"), mice=("participants_id", "nunique"))
counts.loc["total"] = [cells.ephys_roi_id.nunique(), cells.participants_id.nunique()]
expected = pd.DataFrame({"cells": [15, 27, 54, 96], "mice": [7, 8, 13, 28]}, index=["Cerebellum", "Cortex", "Spinal cord", "total"])
print(counts.join(expected, rsuffix="_paper"))
print("all counts match:", bool((counts.values == expected.loc[counts.index].values).all()))

qc = pd.DataFrame({r: {"seal_gohm": (raw[a]["seal"] or {}).get("seal_gohm", np.nan),
                       "seal_median_gohm": (raw[a]["seal"] or {}).get("seal_median_gohm", np.nan),
                       "seal_n_negative": (raw[a]["seal"] or {}).get("seal_n_negative", np.nan),
                       "seal_n_pulses": (raw[a]["seal"] or {}).get("seal_n_pulses", np.nan),
                       "access_mohm": (raw[a]["breakin"] or {}).get("access_mohm", np.nan),
                       "breakin_rin_mohm": (raw[a]["breakin"] or {}).get("breakin_rin_mohm", np.nan)}
                   for r, a in zip(cells.ephys_roi_id, cells.asset_id)}).T.astype(float)
qc["access_frac"] = qc.access_mohm / qc.breakin_rin_mohm
print("\ncells with a cell-attached sweep:", qc.seal_gohm.notna().sum(), "| with a break-in sweep:", qc.access_mohm.notna().sum())
print("seal (GOhm): %.2f to %.2f, median %.2f; seal > 1 GOhm: %d" % (qc.seal_gohm.min(), qc.seal_gohm.max(), qc.seal_gohm.median(), (qc.seal_gohm > 1).sum()))
print("initial access (MOhm): %.1f to %.1f; < 20 MOhm: %d; < 15%% of break-in Rin: %d" %
      (qc.access_mohm.min(), qc.access_mohm.max(), (qc.access_mohm < 20).sum(), (qc.access_frac < 0.15).sum()))
print("test pulses per cell-attached sweep:", qc.seal_n_pulses.value_counts().to_dict(),
      "| cells with a negative pulse resistance:", int((qc.seal_n_negative > 0).sum()),
      "| their IPFX seal (mean over pulses) and median over pulses (GOhm):",
      qc.loc[qc.seal_n_negative > 0, ["seal_gohm", "seal_median_gohm"]].round(2).values.tolist())
print("seal as the median over pulses (GOhm): %.2f to %.2f; > 1 GOhm: %d" %
      (qc.seal_median_gohm.min(), qc.seal_median_gohm.max(), (qc.seal_median_gohm > 1).sum()))
cell_ok = (qc.seal_gohm > 1) & (qc.access_mohm < 20) & (qc.access_frac < 0.15)
print("cells passing all three:", int(cell_ok.sum()), "of", len(qc))
print("by group:", cells.assign(ok=cells.ephys_roi_id.map(cell_ok)).groupby("injection_target_region").ok.sum().to_dict())
cell_ok_med = (qc.seal_median_gohm > 1) & (qc.access_mohm < 20) & (qc.access_frac < 0.15)
print("with the median seal: cells passing all three", int(cell_ok_med.sum()), "| by group:",
      cells.assign(ok=cells.ephys_roi_id.map(cell_ok_med)).groupby("injection_target_region").ok.sum().to_dict())

**Result: reproduced.** The dandiset holds 96 cells (one NWB file each, no recording
deposited twice) from 28 mice: 27 cells from 8 mice labelled from cortex, 15 from 7 from
cerebellum and 54 from 13 from spinal cord, every count equal to the paper's. By IPFX's
test-pulse measures, 80 cells have a seal above 1 GOhm, 92 an initial access resistance
below 20 MOhm, 93 an access resistance below 15% of the break-in input resistance, and 78
pass all three (all 27 cortex, 11 of 15 cerebellum, 40 of 54 spinal cord cells); 2 cells have
no break-in sweep. IPFX's seal is the mean over the six test pulses of the cell-attached
sweep; in 5 cells one pulse gives a negative resistance, and in 2 of them the mean is
negative (-1.82 and -1.84 GOhm) where the median over the pulses is 1.14 and 1.69 GOhm. With
the median, 81 cells have a seal above 1 GOhm and 79 pass all three. These measures are
reported, not used to exclude cells.

## Sweep inclusion (used by the four comparison claims)

The Methods' sweep rule: bridge balance < 20 MOhm and < 15% of input resistance, bias current
within +-100 pA, RMS noise < 0.07 mV over 1.5 ms and < 0.5 mV over 500 ms. The criteria make
it the main reading and the paper's code, which applies no sweep rule, the variant. The table
shows how many cells keep at least one rheobase sweep, and at least one subthreshold long
square, under the whole rule and under each part alone. The bias part uses the current
applied, from the lab notebook: the NWB value where holding is on, 0 pA where it is off
(`lcne.applied_bias`). For the cells that no sweep within +-100 pA leaves out, the table
after it shows holding and autobias on, the target potential (Vcom) and the measured
baseline, to show whether that current was in fact applied.

In [ ]:
rule = {}
for r, a in zip(cells.ephys_roi_id, cells.asset_id):
    c = raw[a]
    ok_, rin = lcne.sweep_rule(c)
    sw = c["sweeps"].assign(methods=ok_.values)
    rule[r] = (sw, rin)
parts = []
for r, (sw, rin) in rule.items():
    s = sw[sw.stimulus.isin(lcne.LONG_SQUARE_SETS) & (sw.pulse_dur_s >= lcne.MIN_PULSE_S)]
    spk = s.sweep_number.isin(list(raw[cells.set_index("ephys_roi_id").asset_id[r]]["spikes"]))
    sub = s.sweep_number.isin(list(raw[cells.set_index("ephys_roi_id").asset_id[r]]["subthresh"]))
    tests = {"all": s.methods, "bias": s.bias_applied_pa.abs() <= lcne.BIAS_MAX_PA,
             "noise": (s.pre_noise_mv < lcne.PRE_NOISE_MAX_MV) & (s.slow_noise_mv < lcne.SLOW_NOISE_MAX_MV),
             "bridge": (s.bridge_balance_mohm < lcne.BRIDGE_MAX_MOHM) & (s.bridge_balance_mohm < lcne.BRIDGE_MAX_FRAC_RIN * rin)}
    row = {"ephys_roi_id": r, "rin_mohm": rin, "sweeps": len(s)}
    for k, m in tests.items():
        row[f"rheo_{k}"] = bool((m & spk).any())
        row[f"sub_{k}"] = bool((m & sub).any())
    row["sweeps_pass"] = int(s.methods.sum())
    parts.append(row)
parts = pd.DataFrame(parts).set_index("ephys_roi_id")
grp = cells.set_index("ephys_roi_id").injection_target_region
summary = parts.filter(like="rheo_").join(grp).groupby("injection_target_region").sum().T
summary["total"] = summary.sum(axis=1)
print("cells keeping >= 1 spiking rheobase sweep:\n", summary)
summary = parts.filter(like="sub_").join(grp).groupby("injection_target_region").sum().T
summary["total"] = summary.sum(axis=1)
print("\ncells keeping >= 1 subthreshold long square:\n", summary)
print("\nRin (MOhm) used for the 15%% bridge rule: %.0f to %.0f, missing for %d cells (no subthreshold sweep passing bias and noise)" %
      (parts.rin_mohm.min(), parts.rin_mohm.max(), parts.rin_mohm.isna().sum()))
bias_cell = ls.groupby("ephys_roi_id").bias_applied_pa.apply(lambda b: (b.abs() <= lcne.BIAS_MAX_PA).any())
print("cells with no long-square sweep within +-100 pA of applied bias:", int((~bias_cell).sum()),
      "| by group:", grp[bias_cell.index[~bias_cell]].value_counts().to_dict())
old_rule = ls.groupby("ephys_roi_id").bias_current_pa.apply(lambda b: (b.abs() <= lcne.BIAS_MAX_PA).any())
newly = sorted(old_rule.index[~old_rule & bias_cell])
print("cells that pass the bias part only once sweeps with holding off count as 0 pA:", newly)
print(ls[ls.ephys_roi_id.isin(newly) & (ls.holding_enable == 0)][["ephys_roi_id", "sweep_number", "stimulus", "amplitude_pa",
      "holding_level_pa", "bias_applied_pa", "v_base"]].round(1).to_string(index=False))
# Is the bias value the current applied? The MIES lab notebook, per sweep: holding on or off, the holding level,
# autobias and its target potential (Vcom), against the measured baseline before the pulse
nb_ids = bias_cell.index[~bias_cell]
chk = ls[ls.ephys_roi_id.isin(nb_ids) & (ls.pulse_dur_s >= lcne.MIN_PULSE_S)].copy()
chk["vbase_minus_vcom"] = chk.v_base - chk.autobias_vcom_mv
per = chk.groupby("ephys_roi_id").agg(group=("ephys_roi_id", lambda x: grp[x.iloc[0]]), sweeps=("sweep_number", "size"),
                                      holding_on=("holding_enable", "sum"), autobias_on=("autobias", "sum"),
                                      bias_min=("bias_applied_pa", "min"), bias_max=("bias_applied_pa", "max"),
                                      vcom_mv=("autobias_vcom_mv", "median"), v_base_mv=("v_base", "median"),
                                      max_abs_vbase_minus_vcom=("vbase_minus_vcom", lambda x: x.abs().max()))
print("\nthe %d cells with no sweep within +-100 pA, long squares of >= 0.8 s:\n" % len(nb_ids), per.round(1).to_string())
rest = ls[~ls.ephys_roi_id.isin(nb_ids) & (ls.pulse_dur_s >= lcne.MIN_PULSE_S)]
print("baseline minus Vcom (mV), median and 95th percentile of |difference|: these cells %.2f, %.2f; other cells %.2f, %.2f" %
      (chk.vbase_minus_vcom.median(), chk.vbase_minus_vcom.abs().quantile(0.95),
       (rest.v_base - rest.autobias_vcom_mv).median(), (rest.v_base - rest.autobias_vcom_mv).abs().quantile(0.95)))
print("Vcom (mV) of these cells: %.0f to %.0f; of the other cells: %.0f to %.0f" %
      (chk.autobias_vcom_mv.min(), chk.autobias_vcom_mv.max(), rest.autobias_vcom_mv.min(), rest.autobias_vcom_mv.max()))

## patchseq-ap-waveform-spinal-vs-cortex and patchseq-ap-waveform-spinal-vs-cerebellum

> The intracellular action potential waveforms of LC-NE neurons projecting to the spinal cord
> differ significantly from those of neurons projecting to isocortex (two-sided Welch's
> t-test, p < 0.01).

> The intracellular action potential waveforms of LC-NE neurons projecting to the spinal cord
> differ significantly from those of neurons projecting to cerebellum (two-sided Welch's
> t-test, p < 0.05).

Waveform PC1 as in the paper's code (`code/spikes.py`): the lowest-amplitude spiking rheobase
sweep with a pulse of at least 0.8 s, the mean of -5 to 10 ms around every -10 mV eFEL peak,
min-max normalised over -2 to 4 ms, PCA over -3 to 6 ms of the mean-centred cells. Main
reading: the rheobase sweep chosen among sweeps passing the Methods' rule, PCA over the
cells that keep one. Variant: the code's, all sweeps, all 96 cells. A third reading,
showing which part of the rule matters, drops only the bias part.

In [ ]:
aid_of = cells.set_index("ephys_roi_id").asset_id

def allowed_sweeps(r, kind):
    """Sweep numbers a reading may use: None for no rule."""
    c = raw[aid_of[r]]
    sw = rule[r][0]
    if kind is None:
        return None
    if kind == "methods":
        return set(sw.sweep_number[sw.methods])
    nb = (sw.pre_noise_mv < lcne.PRE_NOISE_MAX_MV) & (sw.slow_noise_mv < lcne.SLOW_NOISE_MAX_MV)
    if kind == "rin_all":      # the whole rule, with Rin from every subthreshold long square
        nb = nb & (sw.bias_applied_pa.abs() <= lcne.BIAS_MAX_PA)
        rin = lcne.cell_input_resistance(c)
    elif kind == "nobias":     # the rule without its bias part
        rin = lcne.cell_input_resistance(c, sw.loc[nb, "sweep_number"].tolist())
    keep = nb & (sw.bridge_balance_mohm < lcne.BRIDGE_MAX_MOHM) & (sw.bridge_balance_mohm < lcne.BRIDGE_MAX_FRAC_RIN * rin) \
        & ~sw.aborted.fillna(False).astype(bool)
    return set(sw.sweep_number[keep])

def pc1_reading(kind):
    W, chosen = {}, {}
    for r in cells.ephys_roi_id:
        c = raw[aid_of[r]]
        allowed = allowed_sweeps(r, kind)
        sn = lcne.rheobase_sweep(c, allowed)
        if sn is not None:
            W[r] = c["spikes"][sn]
            chosen[r] = sn
    W = pd.DataFrame(W, index=lcne.spike_time_axis()).T
    return lcne.compute_pc1(W), W, chosen

readings = {"main (Methods' sweep rule)": "methods", "variant (code, no sweep rule)": None,
            "Methods' rule, Rin from all subthreshold sweeps": "rin_all", "Methods' rule without the bias part": "nobias"}
pc_results, pc_values, pc_waves, pc_chosen = [], {}, {}, {}
for name, col in readings.items():
    pc, W, chosen = pc1_reading(col)
    pc_values[name], pc_waves[name], pc_chosen[name] = pc, W, chosen
    g = grp[pc.index]
    for other in ["Cortex", "Cerebellum"]:
        w = lcne.welch(pc[g == "Spinal cord"], pc[g == other])
        pc_results.append({"reading": name, "comparison": f"Spinal cord vs {other}", **w})
wt_ids = set(cells.ephys_roi_id[~cells.nwb_genotype.str.contains("Dbh")])
for name, key in [("main, without the 8 wt/wt cells", "main (Methods' sweep rule)"),
                  ("code variant, without the 8 wt/wt cells", "variant (code, no sweep rule)")]:
    pc = pc_values[key]
    pc = pc[~pc.index.isin(wt_ids)]   # PCA not refitted; the cells are left out of the test
    g = grp[pc.index]
    for other in ["Cortex", "Cerebellum"]:
        w = lcne.welch(pc[g == "Spinal cord"], pc[g == other])
        pc_results.append({"reading": name, "comparison": f"Spinal cord vs {other}", **w})
for name, key in [("main, PCA refitted without the 8 wt/wt cells", "main (Methods' sweep rule)"),
                  ("code variant, PCA refitted without the 8 wt/wt cells", "variant (code, no sweep rule)")]:
    W = pc_waves[key]
    pc = lcne.compute_pc1(W[~W.index.isin(wt_ids)])
    pc_values[name] = pc
    g = grp[pc.index]
    for other in ["Cortex", "Cerebellum"]:
        w = lcne.welch(pc[g == "Spinal cord"], pc[g == other])
        pc_results.append({"reading": name, "comparison": f"Spinal cord vs {other}", **w})
pc_results = pd.DataFrame(pc_results)
print(pc_results.round({"mean_a": 3, "mean_b": 3, "t": 3, "df": 2}).to_string())
# which rheobase sweep each reading chose: sweep number and amplitude, main against code
amp = {r: raw[aid_of[r]]["sweeps"].set_index("sweep_number").amplitude_pa for r in cells.ephys_roi_id}
ch = pd.DataFrame({"group": grp,
                   "code_sweep": pd.Series(pc_chosen["variant (code, no sweep rule)"]),
                   "main_sweep": pd.Series(pc_chosen["main (Methods' sweep rule)"])})
ch["code_pa"] = [amp[r].get(s, np.nan) if pd.notna(s) else np.nan for r, s in zip(ch.index, ch.code_sweep)]
ch["main_pa"] = [amp[r].get(s, np.nan) if pd.notna(s) else np.nan for r, s in zip(ch.index, ch.main_sweep)]
both_ch = ch.dropna(subset=["main_sweep"])
print("\nrheobase sweep: cells with one under the code reading", int(ch.code_sweep.notna().sum()), "| under the main reading", len(both_ch),
      "| same sweep in both:", int((both_ch.main_sweep == both_ch.code_sweep).sum()),
      "| different sweep:", int((both_ch.main_sweep != both_ch.code_sweep).sum()))
d_amp = both_ch.main_pa - both_ch.code_pa
print("amplitude, main minus code (pA), cells with a different sweep: median %.0f, range %.0f to %.0f" %
      (d_amp[d_amp != 0].median(), d_amp.min(), d_amp.max()) if (d_amp != 0).any() else "no amplitude differs")
print(ch.astype({"code_sweep": "Int64", "main_sweep": "Int64"}).to_string())

In [ ]:
main_pc = pc_values["main (Methods' sweep rule)"]
code_pc = pc_values["variant (code, no sweep rule)"]
print("cells in the main reading:", len(main_pc), "of 96; dropped (no rheobase sweep passing the rule):",
      grp.drop(main_pc.index).value_counts().to_dict())
both = main_pc.index
print("PC1 of the shared cells, main against code reading: Pearson r = %.3f (signs are arbitrary)" % np.corrcoef(main_pc[both], code_pc[both])[0, 1])
for name, pc in [("main", main_pc), ("code", code_pc)]:
    print(f"{name}: PC1 range {pc.min():.2f} to {pc.max():.2f}")
# by animal
by_mouse = pd.DataFrame({"PC1": main_pc, "group": grp[main_pc.index], "mouse": cells.set_index("ephys_roi_id").participants_id[main_pc.index]})
per_mouse = by_mouse.groupby(["group", "mouse"]).PC1.agg(["mean", "size"]).round(3)
print("\nmain reading, PC1 by mouse:\n", per_mouse.to_string())
mm = by_mouse.groupby(["group", "mouse"]).PC1.mean().reset_index()
for other in ["Cortex", "Cerebellum"]:
    w = lcne.welch(mm.PC1[mm.group == "Spinal cord"], mm.PC1[mm.group == other])
    print(f"mouse means, Spinal cord vs {other}: n = {w['n_a']} and {w['n_b']} mice, t = {w['t']:.2f}, p = {w['p']:.3g}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for k, (name, pc) in enumerate([("main", main_pc), ("code variant", code_pc)]):
    for gname in ["Spinal cord", "Cortex", "Cerebellum"]:
        v = np.sort(pc[grp[pc.index] == gname])
        ax[k].step(v, np.arange(1, len(v) + 1) / len(v), where="post", label=f"{gname} (n={len(v)})")
    ax[k].set(title=f"Waveform PC1, {name}", xlabel="PC1", ylabel="cumulative fraction")
    ax[k].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("all PC1 values are inside the plotted axes (the axes span each reading's full range)")

**Result: reproduced (spinal cord against cortex).** Under the main reading, the Methods'
sweep rule, 69 of the 96 cells keep a rheobase sweep, and waveform PC1 differs between
spinal cord and cortex cells (two-sided Welch t-test, t = -4.18, df = 56.0, p = 0.00011; 36
against 24 cells), below the paper's p < 0.01. The code's variant on all 96 cells gives p =
0.0097 (54 against 27), also below 0.01, and so do the rule with Rin from all subthreshold
sweeps (p = 0.00019) and the rule without its bias part (p = 0.0038). The main reading picks
the same rheobase sweep as the code in 68 of the 69 cells, and in the other one a sweep 10 pA
higher. Over mouse means the difference holds at p = 0.017 (12 against 7 mice). Leaving out
the 8 cells of the two wt/wt spinal cord donors, the main reading still gives p = 0.0056 (p =
0.0067 with the PCA refitted without them), but the code's variant gives p = 0.14 (0.13
refitted): the paper's own computation of this result rests on those cells.

**Result: reproduced (spinal cord against cerebellum).** Under the main reading PC1 differs
between spinal cord and cerebellum cells with p = 0.029 (t = -2.41, df = 14.9; 36 against 9
cells), below the paper's p < 0.05; the code's variant gives p = 0.031 (54 against 15), and the
other two readings p = 0.042 and 0.033. The main reading keeps 9 of the 15 cerebellum cells.
Over mouse means the difference is not significant (p = 0.060, 12 against 5 mice), and
without the 8 cells of the two wt/wt donors it is not significant under either reading (p =
0.19 main, 0.20 code; 0.20 and 0.16 with the PCA refitted without them). None of the 8 wt/wt
cells has a transcriptome (section patchseq-ne-identity), so their identity cannot be checked.

## tau-table-link

The code variant of the tau claims uses the paper's frozen table of IPFX time constants
(LCNE-patchseq-figures at 79fa20a, `code/data/LCNE_patchseq_S14_cell_table.csv`). It is
matched to the dandiset by ephys_roi_id and compared with IPFX's time constant computed
here from all of each cell's subthreshold long squares.

In [ ]:
FROZEN_URL = "https://raw.githubusercontent.com/AllenNeuralDynamics/LCNE-patchseq-figures/79fa20aff7f0195140fac6584962814562c33922/code/data/LCNE_patchseq_S14_cell_table.csv"
frozen = pd.read_csv(io.StringIO(urllib.request.urlopen(FROZEN_URL).read().decode()), dtype={"ephys_roi_id": str})
print("rows:", len(frozen), "| ids matching an NWB file:", frozen.ephys_roi_id.isin(cells.ephys_roi_id).sum(),
      "| files without a row:", (~cells.ephys_roi_id.isin(frozen.ephys_roi_id)).sum(),
      "| missing tau:", frozen.membrane_time_constant_ms.isna().sum())
tau = pd.DataFrame(index=cells.ephys_roi_id)
tau["group"] = grp
tau["mouse"] = cells.set_index("ephys_roi_id").participants_id
tau["frozen"] = frozen.set_index("ephys_roi_id").membrane_time_constant_ms
res_all = {r: lcne.ipfx_tau(raw[aid_of[r]]) for r in tau.index}
tau["ipfx_all"] = [res_all[r][0] for r in tau.index]
tau["n_sweeps_all"] = [res_all[r][1] for r in tau.index]
both = tau.dropna(subset=["frozen", "ipfx_all"])
rel = (both.ipfx_all - both.frozen).abs() / both.frozen
print("cells with a value on both sides:", len(both), "| within 10%%: %d (%.0f%%) | identical to 0.01 ms: %d | Pearson r = %.3f" %
      ((rel <= 0.10).sum(), 100 * (rel <= 0.10).mean(), ((both.ipfx_all - both.frozen).abs() < 0.01).sum(), np.corrcoef(both.ipfx_all, both.frozen)[0, 1]))
import ipfx
print("IPFX version:", ipfx.__version__)
top = rel.sort_values(ascending=False).head(5).index
print("the five largest relative differences:\n", pd.DataFrame({"group": tau.group[top], "frozen_ms": tau.frozen[top], "ipfx_all_ms": tau.ipfx_all[top],
      "relative_diff": rel[top], "sweeps_averaged": tau.n_sweeps_all[top],
      "subthreshold_sweeps_available": [len(raw[aid_of[r]]["subthresh"]) for r in top]}).round(3).to_string())
print("cells with no frozen value but a value here:", tau.frozen.isna().sum() - tau[tau.frozen.isna()].ipfx_all.isna().sum(),
      "| by group:", tau[tau.frozen.isna()].group.value_counts().to_dict())
print("cells with no value here:", tau.ipfx_all.isna().sum(), tau.index[tau.ipfx_all.isna()].tolist())
no_bias = bias_cell.index[~bias_cell]
print("cells with no sweep within +-100 pA of applied bias:", len(no_bias), "| of these with a value in the frozen table:", int(tau.frozen[no_bias].notna().sum()))

**Result: reproduced.** All 96 rows of the frozen table match an NWB file by ephys_roi_id.
Of the 82 cells with a value on both sides, the IPFX (2.1.2) time constant computed here from
all subthreshold long squares is within 10% of the table's for 77 (94%), 24 to 0.01 ms, with
Pearson r = 0.992. The two largest differences are of 41% (cell 1426070420, cortex, 22.2 ms
in the table against 31.4 ms here; cell 1410668927, spinal cord, 8.8 against 12.4 ms), each
from 4 sweeps, all the cell's subthreshold sweeps; the next three are 15% to 19%. The table has
no value for 14 cells; 13 of these have one here. It does have values for 16 of the 18 cells
with no long square within +-100 pA of applied bias, so the frozen values were not computed
under the Methods' bias rule.

## patchseq-tau-cortex-vs-spinal and patchseq-tau-cortex-vs-cerebellum

> LC-NE neurons projecting to isocortex have significantly longer membrane time constants than
> neurons projecting to the spinal cord (two-sided Welch's t-test, p < 0.001).

> LC-NE neurons projecting to isocortex have significantly longer membrane time constants than
> neurons projecting to cerebellum (two-sided Welch's t-test, p < 0.001).

Main reading: IPFX's LongSquareAnalysis time constant at its defaults over the subthreshold
long squares passing the Methods' sweep rule. Variant: the frozen table. Also shown: IPFX on
all sweeps (no rule), and the rule without its bias part.

In [ ]:
def tau_reading(kind, sets=None):
    out = {}
    for r in tau.index:
        c = raw[aid_of[r]]
        allowed = allowed_sweeps(r, kind)
        if sets is not None:
            sw = rule[r][0]
            in_sets = set(sw.sweep_number[sw.stimulus.isin(sets)])
            allowed = in_sets if allowed is None else allowed & in_sets
        out[r] = lcne.ipfx_tau(c, allowed)[0]
    return pd.Series(out)

tau["methods"] = tau_reading("methods")
tau["rin_all"] = tau_reading("rin_all")
tau["nobias"] = tau_reading("nobias")
tau["methods_x1ps"] = tau_reading("methods", ["X1PS_SubThresh_DA_0"])
tau["all_x1ps"] = tau_reading(None, ["X1PS_SubThresh_DA_0"])
tau_results = []
for name, col in [("main (Methods' sweep rule, IPFX)", "methods"), ("variant (code, frozen table)", "frozen"),
                  ("IPFX on all sweeps", "ipfx_all"), ("Methods' rule, Rin from all subthreshold sweeps", "rin_all"),
                  ("Methods' rule without the bias part", "nobias"), ("Methods' rule, X1PS_SubThresh sweeps only", "methods_x1ps"),
                  ("all sweeps, X1PS_SubThresh sweeps only", "all_x1ps")]:
    for other in ["Spinal cord", "Cerebellum"]:
        w = lcne.welch(tau.loc[tau.group == "Cortex", col], tau.loc[tau.group == other, col])
        tau_results.append({"reading": name, "comparison": f"Cortex vs {other}", **w})
wt_ids = set(cells.ephys_roi_id[~cells.nwb_genotype.str.contains("Dbh")])
for other in ["Spinal cord", "Cerebellum"]:
    x = tau[~tau.index.isin(wt_ids)]
    w = lcne.welch(x.loc[x.group == "Cortex", "methods"], x.loc[x.group == other, "methods"])
    tau_results.append({"reading": "main, without the 8 wt/wt cells", "comparison": f"Cortex vs {other}", **w})
tau_results = pd.DataFrame(tau_results)
print(tau_results.round({"mean_a": 2, "mean_b": 2, "t": 3, "df": 2}).to_string())
print("\ncells with a tau, main reading:", tau.methods.notna().groupby(tau.group).sum().to_dict(),
      "| dropped:", tau.methods.isna().groupby(tau.group).sum().to_dict())
for col in ["methods", "frozen", "ipfx_all"]:
    print(f"{col}: tau range {tau[col].min():.1f} to {tau[col].max():.1f} ms")

In [ ]:
per_mouse = tau.groupby(["group", "mouse"])[["methods", "ipfx_all"]].agg(["mean", "count"]).round(1)
print("tau (ms) by mouse:\n", per_mouse.to_string())
mm = tau.groupby(["group", "mouse"]).methods.mean().dropna().reset_index()
for other in ["Spinal cord", "Cerebellum"]:
    w = lcne.welch(mm.methods[mm.group == "Cortex"], mm.methods[mm.group == other])
    print(f"main reading, mouse means, Cortex vs {other}: n = {w['n_a']} and {w['n_b']} mice, means {w['mean_a']:.1f} and {w['mean_b']:.1f} ms, t = {w['t']:.2f}, p = {w['p']:.3g}")
# did the cells dropped by the rule differ? tau on all sweeps for kept and dropped cells
kept = tau.methods.notna()
for gname in ["Cortex", "Spinal cord", "Cerebellum"]:
    m = tau.group == gname
    print(f"{gname}: IPFX tau on all sweeps, kept cells {tau.ipfx_all[m & kept].mean():.1f} ms (n={int((m & kept).sum())}), "
          f"dropped cells {tau.ipfx_all[m & ~kept].mean():.1f} ms (n={int((m & ~kept).sum())})")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for k, col in enumerate(["methods", "frozen"]):
    for gname in ["Spinal cord", "Cortex", "Cerebellum"]:
        v = np.sort(tau.loc[tau.group == gname, col].dropna())
        ax[k].step(v, np.arange(1, len(v) + 1) / len(v), where="post", label=f"{gname} (n={len(v)})")
    ax[k].set(title=f"Membrane time constant, {col}", xlabel="tau (ms)", ylabel="cumulative fraction")
    ax[k].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Result: partially reproduced (cortex against spinal cord).** Under the main reading, IPFX
on the sweeps passing the Methods' rule, cortex cells have the longer time constant (55.4
against 43.1 ms; t = 2.80, df = 42.7), with p = 0.0077 (24 against 36 cells): below 0.05 and
not below the paper's 0.001. The frozen table gives p = 4.0e-05 and IPFX on all sweeps p =
1.3e-04, both below 0.001. The rule drops 18 of the 54 spinal cord cells, 16 of them by its
bias part alone, and these have the shorter time constants (21.8 ms on all sweeps against
43.0 ms for the kept cells); without the bias part p = 0.00055. That part rests on a value the
lab notebook confirms: in every long square of the 18 cells (11 spinal cord, 5 cerebellum, 2
cortex) with no sweep within +-100 pA, I-Clamp holding and autobias are on, and the baseline
before the pulse lies within 1.5 mV of the autobias target (-70 to -41 mV), so the bias beyond
-100 pA was applied to hold these cells there. One spinal cord cell (1392357828) is kept only
because its X1PS_SubThresh sweeps were recorded with holding off (0 pA); its baseline in those
sweeps is -22.7 mV. Over mouse means, main reading, p = 0.0094 (7 against 12 mice). At the cell
level without the 8 wt/wt cells, p = 0.0078.

**Result: partially reproduced (cortex against cerebellum).** Under the main reading cortex
cells have the longer time constant (55.4 against 37.1 ms; t = 3.38, df = 23.7), with p =
0.0025 (24 against 10 cells): below 0.05 and not below 0.001. The frozen table (p = 3.1e-04),
IPFX on all sweeps (p = 2.0e-04) and the rule without its bias part (p = 0.00051) give p below
0.001. The rule drops 5 of the 15 cerebellum cells, with shorter time constants (13.7 against
37.1 ms on all sweeps). Over mouse means, main reading, p = 0.0026 (7 against 5 mice).

## patchseq-transcriptome-link

Each cell's library identifier in `mapping.tsv` is looked up at the NeMO API; its FASTQ
files are streamed into memory and their MD5 sums checked against NeMO's.

In [ ]:
mapping["nemo_lib"] = mapping.nemo_assets_url.str.strip().str.extract(r"(nemo:lib-[a-z0-9]+)")[0]
has_lib = mapping.library_aliquot_nhash_id.notna() & (mapping.library_aliquot_nhash_id.str.strip() != "")
print("cells in mapping.tsv:", len(mapping), "| with a library identifier:", int(has_lib.sum()), "| with a NeMO url:", mapping.nemo_lib.notna().sum())
lib_grp = mapping.assign(ephys_roi_id=mapping.session_id.str.replace("ses-", "")).set_index("ephys_roi_id")
lib_grp["group"] = grp
print("with a library, by group:", lib_grp[has_lib.values].group.value_counts().to_dict(), "of", grp.value_counts().to_dict())
kmer_index, gene_info = rna.gene_kmers()
print("\ngene bodies (mm39, Ensembl 116 coordinates):\n", pd.DataFrame(gene_info).T.to_string())
t = time.time()
libs = mapping.loc[has_lib, "nemo_lib"].tolist()
counted = rna.count_all(libs, kmer_index)
print(f"\ncounted {len(counted)} libraries in {time.time() - t:.0f} s")
rows = []
for _, m in mapping[has_lib].iterrows():
    c = counted[m.nemo_lib]
    if "error" in c:
        rows.append({"ephys_roi_id": m.session_id.replace("ses-", ""), "error": c["error"]})
        continue
    names = [f["library"] for f in c["files"]]
    rows.append({"ephys_roi_id": m.session_id.replace("ses-", ""), "n_files": len(c["files"]),
                 "name_matches": all(n == m.library_aliquot_nhash_id.strip() for n in names),
                 "r1_r2": sorted(f["name"][-11:-9] for f in c["files"]) == ["R1", "R2"],
                 "md5_ok": c["md5_ok"], "pairs": c["n_pairs"], **{f"cpm_{g}": v for g, v in c["cpm"].items()},
                 **{f"n_{g}": v for g, v in c["counts"].items()}})
lib_tab = pd.DataFrame(rows).set_index("ephys_roi_id")
print("libraries whose download failed and was retried:", sum(bool(c.get("retried")) for c in counted.values()),
      "| still failing after the retries:", {k: v["error"] for k, v in counted.items() if "error" in v})
assert not any("error" in c for c in counted.values()), "a library could not be read; its counts would be missing"
print("libraries with two files R1 and R2:", int(lib_tab.r1_r2.sum()), "| NeMO name equals mapping id:", int(lib_tab.name_matches.sum()),
      "| MD5 sums match:", int(lib_tab.md5_ok.sum()), "of", len(lib_tab))
meta_lib = rna.json.loads(rna._get(f"{rna.NEMO_API}/sample/{libs[0]}"))
print("example NeMO sample attributes:", meta_lib["sample_attributes"], "| technique:", counted[libs[0]]["files"] and
      rna.json.loads(rna._get(f"{rna.NEMO_API}/file/{counted[libs[0]]['files'][0]['nemo_id']}"))["technique"])
print("read pairs per library: %d to %d (median %d)" % (lib_tab.pairs.min(), lib_tab.pairs.max(), lib_tab.pairs.median()))
# do the libraries belong to nemo:col-s96tk9a, the collection the claim names?
col = rna.nemo_collection("nemo:col-s96tk9a")
print("\nnemo:col-s96tk9a:", col["name"], "| manifest folder:", col["manifest_urls"])
listing = rna.list_http_dir(col["manifest_urls"][0])
man_name = [n for n, _ in listing if n.endswith("manifest.txt")]
print("manifest file in that folder:", man_name)
man = rna.read_manifest(col["manifest_urls"][0] + man_name[0])
print("manifest rows:", len(man), "| libraries:", man.library_aliquot_nhash_id.nunique())
files_here = pd.DataFrame([{"lib": m.library_aliquot_nhash_id.strip(), "name": f["name"], "md5": f["md5"], "url": f["url"]}
                           for _, m in mapping[has_lib].iterrows() for f in counted[m.nemo_lib].get("files", [])])
in_man = files_here.merge(man, left_on=["lib", "name"], right_on=["library_aliquot_nhash_id", "filename"], how="left")
print("files of the 64 libraries:", len(files_here), "| listed in the manifest by library and file name:", int(in_man.checksum.notna().sum()),
      "| with the same MD5:", int((in_man.checksum == in_man.md5).sum()),
      "| under the collection's manifest folder:", int(files_here.url.str.startswith(col["manifest_urls"][0]).sum()))
print("manifest libraries not among the dandiset's 64:", sorted(set(man.library_aliquot_nhash_id) - set(files_here.lib)))
f0 = rna.json.loads(rna._get(f"{rna.NEMO_API}/file/{counted[libs[0]]['files'][0]['nemo_id']}"))
print("collections the NeMO API lists for a file of these libraries:", [u.rsplit("/", 1)[-1] for u in f0["collections"]],
      "| parent of nemo:col-s96tk9a:", rna.json.loads(rna._get(f"{rna.NEMO_API}/collection/nemo:col-s96tk9a"))["parent_collections"])

In [ ]:
tech = []
for lib in libs:
    c = counted[lib]
    if "error" in c:
        continue
    f = rna.json.loads(rna._get(f"{rna.NEMO_API}/file/{c['files'][0]['nemo_id']}"))
    tech.append((f["technique"], f["aliquot"]["specimen_type"]))
print("technique and specimen type over all libraries:", pd.Series(tech).value_counts().to_dict())

**Result: reproduced.** 64 of the 96 cells have a library identifier in `mapping.tsv` (16 of
27 cortex, 12 of 15 cerebellum, 36 of 54 spinal cord cells). Every one resolves at NeMO to a
library of the same name with two FASTQ files, R1 and R2, whose MD5 sums match NeMO's, all
SMART-seq libraries of nuclei, with 175,250 to 1,110,791 read pairs. All 128 files are listed,
by library and file name and with the same MD5 sum, in the manifest of nemo:col-s96tk9a, the
collection the claim names, and lie under its manifest folder; the manifest lists no other
library. The NeMO API lists the first of these files under a snapshot collection
(nemo:col-nuo6q49), not under nemo:col-s96tk9a. The
other 32 cells have no transcriptome in the deposit.

## patchseq-ne-identity

> The RNA sequenced from each recorded cell's nucleus confirmed that every recorded cell was a
> noradrenergic neuron.

Readings named by the criterion: (a) Dbh detected (CPM > 0); (b) all four of Dbh, Th, Slc6a2
and Slc18a2 detected; (c) the cell's type mapped to the LC-NE type with MapMyCells, which is
not run here. Read pairs are counted by exact 31-mer matches to each gene body (exons and
introns, repeats masked), in place of the paper's STAR alignment to mm10 (Methods, cDNA and
sequencing). Snap25 is a pan-neuronal control; Slc32a1, Tph2 and Alb are controls not
expected in LC-NE neurons.

In [ ]:
cpm = lib_tab.filter(like="cpm_").rename(columns=lambda c: c[4:])
cpm["group"] = grp[cpm.index]
det = cpm[list(rna.GENES)] > 0
print("cells with a transcriptome:", len(cpm), "of 96")
print("detected (CPM > 0), cells per gene:", det.sum().to_dict())
a = det.Dbh
b = det[list(rna.MARKERS)].all(axis=1)
print(f"reading (a), Dbh detected: {int(a.sum())} of {len(a)} ({100 * a.mean():.1f}%)")
print(f"reading (b), all four markers detected: {int(b.sum())} of {len(b)} ({100 * b.mean():.1f}%)")
print("cells failing (b):", cpm.loc[~b, list(rna.MARKERS)].round(1).to_dict("index"))
print("\nCPM by gene (median, min, max):")
print(cpm[list(rna.GENES)].agg(["median", "min", "max"]).round(1).T.to_string())
# a stricter detection threshold, to show the dependence on it
for thr in [1, 10, 100]:
    print(f"CPM > {thr}: Dbh in {int((cpm.Dbh > thr).sum())}, all four markers in {int((cpm[list(rna.MARKERS)] > thr).all(axis=1).sum())}, "
          f"Slc32a1 in {int((cpm.Slc32a1 > thr).sum())}, Tph2 in {int((cpm.Tph2 > thr).sum())}, Alb in {int((cpm.Alb > thr).sum())} of {len(cpm)}")
print("\nby group, reading (b):", b.groupby(cpm.group).agg(["sum", "size"]).to_dict("index"))
print("by mouse, reading (b) fails:", cells.set_index("ephys_roi_id").participants_id[b.index[~b]].value_counts().to_dict())
wt_lib = [r for r in cpm.index if r in wt_ids]
print("\nthe 8 wt/wt cells: with a transcriptome", len(wt_lib), "| reading (a):", int(a[wt_lib].sum()), "| reading (b):", int(b[wt_lib].sum()))
print(cpm.loc[wt_lib, list(rna.GENES)].round(1).join(cells.set_index("ephys_roi_id").participants_id).to_string())

A check of the counting method (verifier's round 1, problem 4). NeMO holds no aligned counts
for the Patch-seq libraries: the SMART-seq v4 STAR 2.7 matrices (collection nemo:col-afgz110) are
of the paper's 1,383 retro-seq nuclei, none of them a Patch-seq library. Those nuclei also have
FASTQ files (nemo:col-3qj1ect), so the k-mer counts can be compared with STAR's on the same
libraries: 60 nuclei that STAR did not exclude, drawn with a fixed seed.

In [ ]:
star, ssv4_samples = rna.star_cpm(list(rna.GENES))
ssv4_samples = ssv4_samples.set_index("exp_component_name")
ok_nuc = sorted(ssv4_samples.index[ssv4_samples["Exclude.STAR"] == "No"])
print("retro-seq nuclei with STAR CPM:", len(star), "| not excluded by STAR QC:", len(ok_nuc))
print("STAR, detected (CPM > 0) among those:", (star.loc[ok_nuc] > 0).sum().to_dict())
rng_rs = np.random.default_rng(1893)
pick = sorted(rng_rs.choice(ok_nuc, 60, replace=False))
t = time.time()
rs = rna.count_files_all(rna.retroseq_files(pick), kmer_index)
print(f"counted {len(rs)} retro-seq libraries in {time.time() - t:.0f} s; retried:", sum(bool(r.get("retried")) for r in rs.values()),
      "| still failing:", {k: v["error"] for k, v in rs.items() if "error" in v},
      "| MD5 sums match the manifest:", sum(r.get("md5_ok", False) for r in rs.values()))
assert not any("error" in r for r in rs.values()), "a retro-seq library could not be read"
km = pd.DataFrame({n: r["cpm"] for n, r in rs.items() if "cpm" in r}).T
st = star.loc[km.index]
cmp_rows = []
for g in rna.GENES:
    both_pos = (km[g] > 0) & (st[g] > 0)
    cmp_rows.append({"gene": g, "kmer_detected": int((km[g] > 0).sum()), "star_detected": int((st[g] > 0).sum()),
                     "detected_by_kmer_only": int(((km[g] > 0) & (st[g] == 0)).sum()),
                     "detected_by_star_only": int(((km[g] == 0) & (st[g] > 0)).sum()),
                     "spearman_r": stats.spearmanr(km[g], st[g])[0] if km[g].nunique() > 1 and st[g].nunique() > 1 else np.nan,
                     "median_ratio_kmer_to_star": float((km[g][both_pos] / st[g][both_pos]).median()) if both_pos.any() else np.nan,
                     "kmer_max": km[g].max(), "star_max": st[g].max()})
print(pd.DataFrame(cmp_rows).set_index("gene").round(3).to_string())
for thr in [0, 10]:
    agree = ((km[list(rna.MARKERS)] > thr).all(axis=1) == (st[list(rna.MARKERS)] > thr).all(axis=1)).sum()
    print(f"all four markers above CPM {thr}: k-mer {int((km[list(rna.MARKERS)] > thr).all(axis=1).sum())}, "
          f"STAR {int((st[list(rna.MARKERS)] > thr).all(axis=1).sum())}, same call in {int(agree)} of {len(km)} nuclei")

**Result: partially reproduced.** Of the 64 cells with a transcriptome, Dbh is detected in 60
(93.8%, reading a) and all four markers in 58 (90.6%, reading b), so neither reading gives
every cell, and both pass the criterion's 90%. One cell (1409801288) has no read of any
marker, with 1,007 Snap25 read pairs; three more lack Dbh. The 32 cells without a
transcriptome cannot be checked, among them all 8 wt/wt cells. The marker counts are high
(median Dbh 192 CPM, Th 710 CPM), and the controls low (Tph2 at most 28 CPM, Alb at most 2.6
CPM, Slc32a1 at most 83 CPM), but at the criterion's threshold of CPM > 0 Slc32a1 is also
detected in 46 of the 64 cells; at CPM > 10, reading (a) gives 58 and reading (b) 55 of 64.
Reading (c) was not run. The k-mer counting agrees with STAR on 60 retro-seq nuclei of the
same paper: Spearman r 0.98 to 0.995 for the four markers and Snap25, the same call on all four
markers for every nucleus, k-mer CPM 0.69 to 0.75 of STAR's, and no k-mer detection of any
marker or control that STAR lacks (Snap25 aside, in 2 nuclei). Slc32a1 is detected by neither
method in those nuclei, so its reads in the Patch-seq libraries are reads those libraries
hold, not false matches of the method; STAR detects Tph2 and Alb at low levels (at most 4.4
and 3.4 CPM) in 10 and 7 of the 60 nuclei where the k-mers find none.

## Blocked claims

Most of the other claims rest on data this dandiset does not hold: whole-brain light-sheet
imaging, single-neuron reconstructions, MAPseq/BARseq, retrograde tracing, MERFISH, in vivo
recordings during behaviour and axonal photometry. The dandiset's metadata lists only
intracellular electrophysiology (variables CurrentClampSeries, VoltageClampSeries and their
stimuli). The sequencing data of the four snRNA-seq and retro-seq claims are at NeMO: the
collection the paper names, col-p9d5w39, holds no files itself but has five child
collections, listed below with what their folders hold.

In [ ]:
import json as _json
meta = _json.loads(urllib.request.urlopen(f"{lcne.API}/dandisets/{lcne.DANDISET_ID}/versions/{lcne.DANDISET_VERSION}/").read())
print("approach:", [x["name"] for x in meta["assetsSummary"].get("approach", [])])
print("variables measured:", meta["assetsSummary"].get("variableMeasured"))
print("related resources:", [(r.get("identifier"), r.get("relation")) for r in meta.get("relatedResource", [])])
top = rna.nemo_collection("nemo:col-p9d5w39")
print("\nNeMO", top["id"], "-", top["name"], "| manifest folders:", top["manifest_urls"], "| child collections:", top["children"])
for ch_id in top["children"]:
    ch = rna.nemo_collection(ch_id)
    print(f"\n{ch_id}: {ch['name']}")
    for u in ch["manifest_urls"]:
        entries = rna.list_http_dir(u)
        print("  ", u, "->", len(entries), "entries")
        folders = [n for n, _ in entries if n.endswith("/")]
        if len(folders) > 8:
            print(f"     {len(folders)} folders, one per library; the first, {folders[0]}, holds {rna.list_http_dir(u + folders[0])}")
        for name, size in entries:
            if name.endswith("/") and len(folders) > 8:
                continue
            if name.endswith("/"):
                sub = rna.list_http_dir(u + name)
                big = [(n, s) for n, s in sub if s.endswith("G")]
                print(f"     {name}: {len(sub)} entries; " + (f"{len(big)} of 1 GB or more, e.g. {big[:2]}" if big else f"e.g. {sub[:3]}"))
            elif len(entries) <= 8:
                print(f"     {name} {size}")

# the per-library 10x count files of the snRNA-seq atlas: their size and shape, read lazily
import h5py, remfile
cnt_url = "https://data.nemoarchive.org/bican/grant/BICAN_Neuromodulation/aibs/transcriptome/nuclei/10x_v4/house_mouse/processed/counts/"
h5ads = [(n, s) for n, s in rna.list_http_dir(cnt_url) if n.endswith(".h5ad")]
shapes = []
for n, s in h5ads:
    with h5py.File(remfile.File(cnt_url + n), "r") as h:
        shapes.append((n, s, h["X"].attrs["encoding-type"], tuple(int(x) for x in h["X"].attrs["shape"]), sorted(h["obs"].keys())[:3]))
print("\n10x h5ad count files:", len(h5ads))
for row in shapes:
    print("  ", row[0], row[1], row[2], "barcodes x genes", row[3])
print("barcodes over all files:", sum(r[3][0] for r in shapes), "| R matrices of all libraries:",
      [(n, s) for n, s in rna.list_http_dir(cnt_url) if n.endswith(".rda")])
print(f"\nnotebook run time: {(time.time() - T0) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| patchseq-sample | reproduced |
| patchseq-transcriptome-link (link) | reproduced |
| patchseq-ne-identity | partially reproduced (Dbh in 60 of 64 cells with a transcriptome, all four markers in 58) |
| tau-table-link (link) | reproduced |
| patchseq-ap-waveform-spinal-vs-cortex | reproduced |
| patchseq-ap-waveform-spinal-vs-cerebellum | reproduced (weak: not significant over mouse means or without the 8 wt/wt cells) |
| patchseq-tau-cortex-vs-spinal | partially reproduced (p < 0.05, not < 0.001, under the Methods' sweep rule) |
| patchseq-tau-cortex-vs-cerebellum | partially reproduced (same) |
| 19 other claims | blocked, data_missing: their data are not in the dandiset |
| snrnaseq-single-cluster | blocked, methods_underspecified: silhouette at K = 1 is undefined |
| pseudospace-graded-genes, merfish-pseudospace-dorsal-ventral, retroseq-projection-gene-space | blocked, not_runnable: the paper's R selection of LC-NE nuclei |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The Methods' sweep rule is applied as written
(main reading), with IPFX's windows for the noise measures, the input resistance from the
sweeps that pass the rule's other parts, and the bias current the amplifier applied, from
the lab notebook (0 pA where holding was off). The paper's code applies no sweep rule, and
the paper's frozen time constants include cells whose every sweep has an applied bias beyond
+-100 pA, so the paper does not appear to have applied that part. The time constant is IPFX's
own functions run on the dandiset's sweeps, with the long squares taken to be the four 1 s
step sets. Marker genes are counted by exact 31-mer matches to gene bodies, not by STAR
alignment to mm10; on 60 retro-seq nuclei the two agree closely.

What was found about the dataset. The cells, mice and groups match the paper exactly; the
transcriptomes are linked by identifier for 64 of the 96 cells and are absent for the rest;
two of the 13 spinal cord donors (8 cells) are C57BL6J wild-type mice by their NWB genotype
and sample names, where the Methods describe Dbh-Cre;Ai65 mice, and none of the 8 has a
transcriptome; the NWB files record no filter setting (the field reads "PLACEHOLDER"); one
subject in `participants.tsv` has no sex or age; `datasets.tsv` gives the URL of dandiset
001748 for dandiset 001858. The NeMO collection the paper names (col-p9d5w39) holds no files
itself; its five child collections hold the Patch-seq, snRNA-seq and retro-seq data, and the
NeMO API lists the Patch-seq file it was asked about under a snapshot collection, not under
col-s96tk9a.